In [1]:
{
  "cells": [
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "from pathlib import Path",
        "import zipfile",
        "import pandas as pd",
        "import numpy as np",
        "",
        "input_root = Path('/kaggle/input')",
        "dataset_dirs = [p for p in input_root.iterdir() if p.is_dir()]",
        "print('Dataset directories:')",
        "for d in dataset_dirs:",
        "    print('-', d)",
        "",
        "zip_files = sorted(input_root.rglob('*.zip'))",
        "print('\\nZip files found:')",
        "for z in zip_files:",
        "    print(f'- {z} | {z.stat().st_size/(1024**3):.2f} GB')"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "assert len(zip_files) > 0, 'No zip file found in /kaggle/input'",
        "",
        "small_zip = None",
        "full_zip = None",
        "for z in zip_files:",
        "    name = z.name.lower()",
        "    if '5%' in name or '5' in name and small_zip is None:",
        "        small_zip = z",
        "    if 'onedrive_2026-07-25' in name and full_zip is None:",
        "        full_zip = z",
        "",
        "if small_zip is None:",
        "    small_zip = min(zip_files, key=lambda p: p.stat().st_size)",
        "if full_zip is None:",
        "    full_zip = max(zip_files, key=lambda p: p.stat().st_size)",
        "",
        "print('Small zip:', small_zip)",
        "print('Full zip :', full_zip)",
        "",
        "with zipfile.ZipFile(small_zip, 'r') as zf:",
        "    members = zf.namelist()",
        "    print('\\nSmall zip total files:', len(members))",
        "    for m in members:",
        "        print('-', m)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "with zipfile.ZipFile(small_zip, 'r') as zf:",
        "    members = zf.namelist()",
        "    all_feature_csv = [m for m in members if 'All features' in m and m.lower().endswith('.csv')]",
        "    best10_csv = [m for m in members if '10-best features' in m and m.lower().endswith('.csv')]",
        "",
        "print('All-features CSV files:')",
        "for m in all_feature_csv:",
        "    print('-', m)",
        "",
        "print('\\n10-best CSV files:')",
        "for m in best10_csv:",
        "    print('-', m)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "preview_parts = []",
        "with zipfile.ZipFile(small_zip, 'r') as zf:",
        "    for m in all_feature_csv:",
        "        with zf.open(m) as f:",
        "            part = pd.read_csv(f, nrows=50000, low_memory=False)",
        "            preview_parts.append(part)",
        "",
        "preview_df = pd.concat(preview_parts, ignore_index=True)",
        "print('Preview shape:', preview_df.shape)",
        "print('Columns count:', len(preview_df.columns))",
        "print('First 20 columns:', preview_df.columns[:20].tolist())",
        "preview_df.head(5)"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "target_candidates = ['label', 'Label', 'attack_cat', 'Attack', 'class', 'Class', 'category']",
        "target_col = next((c for c in target_candidates if c in preview_df.columns), None)",
        "",
        "print('Detected target column:', target_col)",
        "",
        "row_count = 0",
        "missing_sum = None",
        "target_counts = {}",
        "",
        "with zipfile.ZipFile(small_zip, 'r') as zf:",
        "    for m in all_feature_csv:",
        "        with zf.open(m) as f:",
        "            for chunk in pd.read_csv(f, chunksize=200000, low_memory=False):",
        "                row_count += len(chunk)",
        "                if missing_sum is None:",
        "                    missing_sum = pd.Series(0, index=chunk.columns, dtype='int64')",
        "                missing_sum += chunk.isna().sum()",
        "",
        "                if target_col is not None:",
        "                    vc = chunk[target_col].value_counts(dropna=False)",
        "                    for k, v in vc.items():",
        "                        key = str(k)",
        "                        target_counts[key] = target_counts.get(key, 0) + int(v)",
        "",
        "print('Total rows processed:', row_count)",
        "",
        "missing_df = pd.DataFrame({",
        "    'column': missing_sum.index,",
        "    'missing_count': missing_sum.values,",
        "    'missing_ratio': missing_sum.values / row_count",
        "}).sort_values('missing_ratio', ascending=False)",
        "",
        "print('\\nTop missing columns:')",
        "display(missing_df.head(20))",
        "",
        "if target_col is not None:",
        "    target_df = pd.DataFrame({",
        "        'target': list(target_counts.keys()),",
        "        'count': list(target_counts.values())",
        "    }).sort_values('count', ascending=False)",
        "    print('\\nTarget distribution:')",
        "    display(target_df.head(20))"
      ]
    },
    {
      "cell_type": "code",
      "metadata": {
        "language": "python"
      },
      "source": [
        "sample_out = '/kaggle/working/sample_2_percent.csv'",
        "rng = np.random.default_rng(42)",
        "first_write = True",
        "sample_frac = 0.02",
        "total_in = 0",
        "total_out = 0",
        "",
        "with zipfile.ZipFile(small_zip, 'r') as zf:",
        "    for m in all_feature_csv:",
        "        with zf.open(m) as f:",
        "            for chunk in pd.read_csv(f, chunksize=200000, low_memory=False):",
        "                total_in += len(chunk)",
        "                mask = rng.random(len(chunk)) < sample_frac",
        "                sampled = chunk.loc[mask]",
        "                total_out += len(sampled)",
        "                sampled.to_csv(sample_out, mode='w' if first_write else 'a', index=False, header=first_write)",
        "                first_write = False",
        "",
        "print('Sample saved:', sample_out)",
        "print('Input rows   :', total_in)",
        "print('Sample rows  :', total_out)"
      ]
    }
  ]
}

{'cells': [{'cell_type': 'code',
   'metadata': {'language': 'python'},
   'source': ['from pathlib import Path',
    'import zipfile',
    'import pandas as pd',
    'import numpy as np',
    '',
    "input_root = Path('/kaggle/input')",
    'dataset_dirs = [p for p in input_root.iterdir() if p.is_dir()]',
    "print('Dataset directories:')",
    'for d in dataset_dirs:',
    "    print('-', d)",
    '',
    "zip_files = sorted(input_root.rglob('*.zip'))",
    "print('\\nZip files found:')",
    'for z in zip_files:',
    "    print(f'- {z} | {z.stat().st_size/(1024**3):.2f} GB')"]},
  {'cell_type': 'code',
   'metadata': {'language': 'python'},
   'source': ["assert len(zip_files) > 0, 'No zip file found in /kaggle/input'",
    '',
    'small_zip = None',
    'full_zip = None',
    'for z in zip_files:',
    '    name = z.name.lower()',
    "    if '5%' in name or '5' in name and small_zip is None:",
    '        small_zip = z',
    "    if 'onedrive_2026-07-25' in name and full_zip 

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

print("Python OK:", sys.version.split()[0])
print("Pandas OK:", pd.__version__)
print("Numpy OK :", np.__version__)

print("\nInput folders:")
for p in os.listdir("/kaggle/input"):
    print("-", p)

In [3]:
import os, time
t0 = time.time()
print("alive")
print("input_exists:", os.path.exists("/kaggle/input"))
print("items:", len(os.listdir("/kaggle/input")) if os.path.exists("/kaggle/input") else 0)
print("elapsed_sec:", round(time.time() - t0, 2))

alive
input_exists: True
items: 1
elapsed_sec: 0.0


In [4]:
from pathlib import Path

input_root = Path("/kaggle/input")
zip_files = sorted(input_root.rglob("*.zip"))
csv_files = sorted(input_root.rglob("*.csv"))

print("Top-level items:")
for p in input_root.iterdir():
    print("-", p)

print("\nZip count:", len(zip_files))
print("CSV count:", len(csv_files))

print("\nFirst 40 CSV paths:")
for p in csv_files[:40]:
    print("-", p)

Top-level items:
- /kaggle/input/datasets

Zip count: 0
CSV count: 82

First 40 CSV paths:
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/10-best Training-Testing split/UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/10-best Training-Testing split/UNSW_2018_IoT_Botnet_Final_10_best_Training.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/UNSW_2018_IoT_Botnet_Final_10_Best.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_1.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_2.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_3.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_4.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-a

In [5]:
from pathlib import Path

input_root = Path("/kaggle/input")
csv_files = sorted(input_root.rglob("*.csv"))

all_feature_csv = [
    p for p in csv_files
    if "5%" in str(p) and "All features" in str(p)
]

print("Selected all-feature files:", len(all_feature_csv))
for p in all_feature_csv:
    print("-", p)

Selected all-feature files: 4
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_1.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_2.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_3.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_4.csv


In [6]:
import pandas as pd
import numpy as np

preview_parts = []
for p in all_feature_csv:
    part = pd.read_csv(p, nrows=50000, low_memory=False)
    preview_parts.append(part)

preview_df = pd.concat(preview_parts, ignore_index=True)

print("Preview shape:", preview_df.shape)
print("Total columns:", len(preview_df.columns))
print("First 25 columns:")
print(preview_df.columns[:25].tolist())

preview_df.head(5)

Preview shape: (200000, 46)
Total columns: 46
First 25 columns:
['pkSeqID', 'stime', 'flgs', 'flgs_number', 'proto', 'proto_number', 'saddr', 'sport', 'daddr', 'dport', 'pkts', 'bytes', 'state', 'state_number', 'ltime', 'seq', 'dur', 'mean', 'stddev', 'sum', 'min', 'max', 'spkts', 'dpkts', 'sbytes']


,pkSeqID,stime,flgs,flgs_number,proto,proto_number,saddr,sport,daddr,dport,...,AR_P_Proto_P_DstIP,N_IN_Conn_P_DstIP,N_IN_Conn_P_SrcIP,AR_P_Proto_P_Sport,AR_P_Proto_P_Dport,Pkts_P_State_P_Protocol_P_DestIP,Pkts_P_State_P_Protocol_P_SrcIP,attack,category,subcategory
0,1,1.528089e+09,e,1,tcp,1,192.168.100.147,49960,192.168.100.7,80,...,1.12704,96,75,1.133720,1.129970,770,602,1,DoS,HTTP
1,2,1.528089e+09,e,1,arp,2,192.168.100.7,-1,192.168.100.147,-1,...,15267.20000,1,2,0.005142,0.005142,2,6,1,DoS,HTTP
2,3,1.528089e+09,e,1,tcp,1,192.168.100.147,49962,192.168.100.7,80,...,1.12704,96,75,1.135100,1.129970,770,602,1,DoS,HTTP
3,4,1.528089e+09,e,1,tcp,1,192.168.100.147,49964,192.168.100.7,80,...,1.12704,96,75,1.135140,1.129970,770,602,1,DoS,HTTP
4,5,1.528089e+09,e,1,tcp,1,192.168.100.147,49966,192.168.100.7,80,...,1.12704,96,75,1.135260,1.129970,770,602,1,DoS,HTTP


In [7]:
target_candidates = ["label", "Label", "attack_cat", "Attack", "class", "Class", "category"]
target_col = next((c for c in target_candidates if c in preview_df.columns), None)
print("Detected target column:", target_col)

# Missing ratio
missing_ratio = preview_df.isna().mean().sort_values(ascending=False)
missing_df = pd.DataFrame({
    "column": missing_ratio.index,
    "missing_ratio": missing_ratio.values
})

print("\nTop missing columns:")
display(missing_df.head(20))

# Numeric summary
print("\nNumeric summary:")
display(preview_df.describe(include=[np.number]).T)

# Target distribution
if target_col is not None:
    target_df = preview_df[target_col].value_counts(dropna=False).reset_index()
    target_df.columns = ["target", "count"]
    print("\nTarget distribution:")
    display(target_df.head(20))

Detected target column: category

Top missing columns:


,column,missing_ratio
0,pkSeqID,0.0
1,stime,0.0
2,flgs,0.0
3,flgs_number,0.0
4,proto,0.0
5,proto_number,0.0
6,saddr,0.0
7,sport,0.0
8,daddr,0.0
9,dport,0.0



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
pkSeqID,200000.0,1.525000e+06,1.118130e+06,1.000000e+00,7.625008e+05,1.525000e+06,2.287500e+06,3.050000e+06
stime,200000.0,1.528090e+09,7.386124e+03,1.528081e+09,1.528085e+09,1.528092e+09,1.528097e+09,1.528099e+09
flgs_number,200000.0,1.565160e+00,7.310450e-01,1.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,6.000000e+00
proto_number,200000.0,2.000035e+00,9.999800e-01,1.000000e+00,1.000000e+00,2.000000e+00,3.000000e+00,3.000000e+00
sport,200000.0,2.583808e+04,1.860432e+04,-1.000000e+00,9.649000e+03,2.043250e+04,4.006700e+04,6.553500e+04
dport,200000.0,7.999635e+01,5.433537e-01,-1.000000e+00,8.000000e+01,8.000000e+01,8.000000e+01,8.000000e+01
pkts,200000.0,8.369290e+00,3.597110e+00,2.000000e+00,5.000000e+00,7.000000e+00,1.100000e+01,5.600000e+01
bytes,200000.0,7.287995e+02,1.949511e+02,1.200000e+02,6.160000e+02,7.700000e+02,8.400000e+02,6.999000e+03
state_number,200000.0,3.165000e+00,1.071560e+00,1.000000e+00,3.000000e+00,4.000000e+00,4.000000e+00,7.000000e+00
ltime,200000.0,1.528090e+09,7.378920e+03,1.528081e+09,1.528085e+09,1.528093e+09,1.528097e+09,1.528099e+09



Target distribution:


,target,count
0,DoS,100000
1,DDoS,100000


In [10]:
# Cell 1: Install high-performance libraries
!pip -q install polars pyarrow plotly

In [11]:
# Cell 2: Imports and config
from pathlib import Path
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import time
import warnings

warnings.filterwarnings("ignore")

TARGET_SAMPLE_ROWS = 1_200_000   # tune: 800k to 2M
RANDOM_SEED = 42
SAVE_REPORTS = True

t0 = time.time()

In [12]:
# Cell 3: Discover CSV files from Kaggle input
input_root = Path("/kaggle/input")
csv_files = sorted(input_root.rglob("*.csv"))

print("CSV files found:", len(csv_files))
for p in csv_files[:20]:
    print("-", p)

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in /kaggle/input")

CSV files found: 82
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/10-best Training-Testing split/UNSW_2018_IoT_Botnet_Final_10_best_Testing.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/10-best Training-Testing split/UNSW_2018_IoT_Botnet_Final_10_best_Training.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/10-best features/UNSW_2018_IoT_Botnet_Final_10_Best.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_1.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_2.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_3.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/5/5%/All features/UNSW_2018_IoT_Botnet_Full5pc_4.csv
- /kaggle/input/datasets/habib2021/iotbot16-71gb-and-5/OneDrive_2026-07-25/Entire Dataset/UNSW_2018_IoT_Botnet_Dataset_1.

In [14]:
# Cell 4: Fast adaptive sampling across all files
rows_per_file = max(5000, TARGET_SAMPLE_ROWS // max(1, len(csv_files)))
print("Rows per file for sampling:", rows_per_file)

sample_parts = []
for i, f in enumerate(csv_files, 1):
    try:
        part = pl.read_csv(
            f,
            n_rows=rows_per_file,
            infer_schema_length=2000,
            ignore_errors=True,
            null_values=["", "NA", "N/A", "null", "None", "-"]
        )
        part = part.with_columns(pl.lit(str(f)).alias("_source_file"))
        sample_parts.append(part)
    except Exception as e:
        print("Skip:", f, "|", str(e)[:120])

if len(sample_parts) == 0:
    raise RuntimeError("Could not load any CSV sample")

df = pl.concat(sample_parts, how="diagonal_relaxed").rechunk()

print("Sampled shape:", df.shape)
print("Columns:", len(df.columns))
print("Elapsed sec:", round(time.time() - t0, 2))

Rows per file for sampling: 14634
Sampled shape: (1185354, 1036)
Columns: 1036
Elapsed sec: 70.34


In [15]:
# Cell 5: Basic overview
schema_df = pl.DataFrame({
    "column": list(df.schema.keys()),
    "dtype": [str(v) for v in df.schema.values()]
})

print("First 30 columns:")
print(schema_df.head(30))
print("\nHead:")
print(df.head(5))

First 30 columns:
shape: (30, 2)
┌─────────┬────────┐
│ column  ┆ dtype  │
│ ---     ┆ ---    │
│ str     ┆ str    │
╞═════════╪════════╡
│ pkSeqID ┆ String │
│ proto   ┆ String │
│ saddr   ┆ String │
│ sport   ┆ String │
│ daddr   ┆ String │
│ …       ┆ …      │
│ pkts    ┆ String │
│ bytes   ┆ String │
│ state   ┆ String │
│ ltime   ┆ String │
│ dur     ┆ String │
└─────────┴────────┘

Head:
shape: (5, 1_036)
┌─────────┬───────┬─────────────────┬───────┬───┬──────┬──────┬──────┬──────────────┐
│ pkSeqID ┆ proto ┆ saddr           ┆ sport ┆ … ┆ doui ┆ sco  ┆ dco  ┆ subcategory  │
│ ---     ┆ ---   ┆ ---             ┆ ---   ┆   ┆ ---  ┆ ---  ┆ ---  ┆ ---          │
│ str     ┆ str   ┆ str             ┆ str   ┆   ┆ str  ┆ str  ┆ str  ┆ str          │
╞═════════╪═══════╪═════════════════╪═══════╪═══╪══════╪══════╪══════╪══════════════╡
│ 792371  ┆ udp   ┆ 192.168.100.150 ┆ 48516 ┆ … ┆ null ┆ null ┆ null ┆ null         │
│ 2056418 ┆ tcp   ┆ 192.168.100.148 ┆ 22267 ┆ … ┆ null ┆ null ┆ null 

In [16]:
# Cell 6: Missingness, uniqueness, and sentinel checks
n = df.height

summary_rows = []
for c in df.columns:
    s = df.get_column(c)
    null_count = s.null_count()
    null_ratio = null_count / n if n else 0.0
    
    try:
        unique_count = s.n_unique()
    except:
        unique_count = None
    
    sentinel_neg1 = None
    sentinel_zero = None
    
    if s.dtype in [pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64]:
        sentinel_neg1 = int((s == -1).sum())
        sentinel_zero = int((s == 0).sum())
    
    summary_rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "null_count": int(null_count),
        "null_ratio": float(null_ratio),
        "unique_count": None if unique_count is None else int(unique_count),
        "count_-1": sentinel_neg1,
        "count_0": sentinel_zero
    })

quality_df = pl.DataFrame(summary_rows).sort("null_ratio", descending=True)
print("Top missing columns:")
print(quality_df.select(["column", "dtype", "null_ratio", "unique_count"]).head(20))

Top missing columns:
shape: (20, 4)
┌───────────────┬────────┬────────────┬──────────────┐
│ column        ┆ dtype  ┆ null_ratio ┆ unique_count │
│ ---           ┆ ---    ┆ ---        ┆ ---          │
│ str           ┆ str    ┆ f64        ┆ i64          │
╞═══════════════╪════════╪════════════╪══════════════╡
│ _duplicated_1 ┆ String ┆ 1.0        ┆ 1            │
│ _duplicated_2 ┆ String ┆ 1.0        ┆ 1            │
│ _duplicated_3 ┆ String ┆ 1.0        ┆ 1            │
│ _duplicated_4 ┆ String ┆ 1.0        ┆ 1            │
│ _duplicated_5 ┆ String ┆ 1.0        ┆ 1            │
│ …             ┆ …      ┆ …          ┆ …            │
│ 48765         ┆ String ┆ 0.987659   ┆ 13150        │
│ 47869         ┆ Int64  ┆ 0.987659   ┆ 13565        │
│ 52526         ┆ Int64  ┆ 0.987658   ┆ 14467        │
│ 31890         ┆ Int64  ┆ 0.987657   ┆ 12199        │
│ 7564          ┆ Int64  ┆ 0.987657   ┆ 14578        │
└───────────────┴────────┴────────────┴──────────────┘


In [17]:
# Cell 7: Numeric descriptive stats
num_cols = [c for c, t in df.schema.items() if t in {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64
}]

if len(num_cols) > 0:
    num_stats = df.select(num_cols).describe(percentiles=[0.25, 0.5, 0.75]).transpose(
        include_header=True, header_name="metric"
    )
    print("Numeric stats:")
    print(num_stats.head(50))
else:
    print("No numeric columns found.")

Numeric stats:
shape: (50, 10)
┌────────────┬──────────┬────────────┬────────────┬───┬──────────┬──────────┬──────────┬───────────┐
│ metric     ┆ column_0 ┆ column_1   ┆ column_2   ┆ … ┆ column_5 ┆ column_6 ┆ column_7 ┆ column_8  │
│ ---        ┆ ---      ┆ ---        ┆ ---        ┆   ┆ ---      ┆ ---      ┆ ---      ┆ ---       │
│ str        ┆ str      ┆ str        ┆ str        ┆   ┆ str      ┆ str      ┆ str      ┆ str       │
╞════════════╪══════════╪════════════╪════════════╪═══╪══════════╪══════════╪══════════╪═══════════╡
│ statistic  ┆ count    ┆ null_count ┆ mean       ┆ … ┆ 25%      ┆ 50%      ┆ 75%      ┆ max       │
│ N_IN_Conn_ ┆ 87804.0  ┆ 1097550.0  ┆ 83.9024759 ┆ … ┆ 71.0     ┆ 100.0    ┆ 100.0    ┆ 100.0     │
│ P_SrcIP    ┆          ┆            ┆ 6920413    ┆   ┆          ┆          ┆          ┆           │
│ state_numb ┆ 87804.0  ┆ 1097550.0  ┆ 3.18841966 ┆ … ┆ 3.0      ┆ 4.0      ┆ 4.0      ┆ 7.0       │
│ er         ┆          ┆            ┆ 19743976   ┆   ┆     

In [18]:
# Cell 8: Target detection + class distribution
target_candidates = ["label", "Label", "attack", "Attack", "class", "Class", "attack_cat", "category", "Category"]
target_col = next((c for c in target_candidates if c in df.columns), None)

print("Detected target column:", target_col)

if target_col is not None:
    target_dist = (
        df.group_by(target_col)
          .count()
          .rename({"count": "rows"})
          .sort("rows", descending=True)
    )
    print(target_dist.head(20))
    
    fig = px.bar(
        target_dist.to_pandas().head(20),
        x=target_col,
        y="rows",
        title=f"Top class distribution: {target_col}",
    )
    fig.show()
else:
    print("No known target column found from candidate list.")

Detected target column: attack
shape: (3, 2)
┌────────┬─────────┐
│ attack ┆ rows    │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ null   ┆ 1097550 │
│ 1      ┆ 87796   │
│ 0      ┆ 8       │
└────────┴─────────┘


In [19]:
# Cell 9: High-level visual EDA
# 9.1 Missing ratio bar chart (top 20)
miss_pd = quality_df.select(["column", "null_ratio"]).to_pandas().head(20)
fig1 = px.bar(miss_pd, x="column", y="null_ratio", title="Top 20 Missing Ratio Columns")
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

# 9.2 Correlation heatmap (top numeric columns)
if len(num_cols) >= 2:
    top_num = num_cols[:20]
    corr_pd = df.select(top_num).to_pandas().corr(numeric_only=True)
    fig2 = px.imshow(corr_pd, title="Correlation Heatmap (Top Numeric Columns)", aspect="auto")
    fig2.show()
else:
    print("Not enough numeric columns for correlation heatmap.")

In [20]:
# Cell 10: Save EDA reports
if SAVE_REPORTS:
    out_dir = Path("/kaggle/working/eda_fast_outputs")
    out_dir.mkdir(parents=True, exist_ok=True)

    quality_df.write_csv(out_dir / "quality_summary.csv")
    schema_df.write_csv(out_dir / "schema_summary.csv")

    if target_col is not None:
        target_dist.write_csv(out_dir / "target_distribution.csv")

    if len(num_cols) > 0:
        df.select(num_cols).describe(percentiles=[0.25, 0.5, 0.75]).write_csv(out_dir / "numeric_describe.csv")

    print("Saved reports to:", out_dir)

print("Total elapsed sec:", round(time.time() - t0, 2))

Saved reports to: /kaggle/working/eda_fast_outputs
Total elapsed sec: 362.63


In [24]:
!pip -q install polars pyarrow plotly seaborn scipy

In [26]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

START_TIME = time.time()

# Speed vs depth controls
FAST_MODE = True
SAMPLE_ROWS_PER_FILE = 200_000 if FAST_MODE else 500_000
MAX_NUMERIC_PLOT_COLS = 12
TOP_K_CATEGORIES = 15
CORR_TOP_N = 30
HIGH_CORR_THRESHOLD = 0.90
REPORT_DIR = Path("/kaggle/working/eda_full_outputs")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

In [27]:
import re
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

if "df" not in globals():
    raise RuntimeError("df not found. Run previous loading cells first.")

print("Original shape:", df.shape)

bad_name_cols = []
for c in df.columns:
    c_low = c.lower()
    if c.startswith("_duplicated_") or c_low.startswith("unnamed") or re.fullmatch(r"\d+(\.\d+)?", c):
        bad_name_cols.append(c)

high_null_cols = []
for c in df.columns:
    nr = df.get_column(c).null_count() / max(df.height, 1)
    if nr > 0.98:
        high_null_cols.append(c)

drop_cols = sorted(set(bad_name_cols + high_null_cols))
df_clean = df.drop(drop_cols) if drop_cols else df.clone()

text_keep = {
    "_source_file", "proto", "state", "flgs", "saddr", "daddr",
    "category", "subcategory", "attack", "Attack", "label", "Label",
    "class", "Class", "attack_cat"
}

cast_expr = []
for c in df_clean.columns:
    if c in text_keep:
        cast_expr.append(pl.col(c))
    else:
        cast_expr.append(pl.col(c).cast(pl.Float64, strict=False).alias(c))

df_work = df_clean.select(cast_expr).rechunk()

print("Dropped columns:", len(drop_cols))
print("Cleaned shape:", df_work.shape)
print("Sample dropped cols:", drop_cols[:20])

Original shape: (1185354, 1036)
Dropped columns: 935
Cleaned shape: (1185354, 101)
Sample dropped cols: ['', '0', '0.000000', '0.000002', '0.000004', '0.000006', '0.000007', '0.000011', '0.000836', '0.000836_duplicated_0', '0.002508', '0.002713', '0.006463', '0.007054', '0.007098', '0.010286', '0.017158', '0.022604', '0.024266', '0.036398']


In [28]:
quality_rows = []
for c in df_work.columns:
    s = df_work.get_column(c)
    quality_rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "null_count": int(s.null_count()),
        "null_ratio": float(s.null_count() / max(df_work.height, 1)),
        "n_unique": int(s.n_unique())
    })

quality_df2 = pl.DataFrame(quality_rows).sort("null_ratio", descending=True)
print(quality_df2.head(25))

shape: (25, 5)
┌───────────────────┬─────────┬────────────┬────────────┬──────────┐
│ column            ┆ dtype   ┆ null_count ┆ null_ratio ┆ n_unique │
│ ---               ┆ ---     ┆ ---        ┆ ---        ┆ ---      │
│ str               ┆ str     ┆ i64        ┆ f64        ┆ i64      │
╞═══════════════════╪═════════╪════════════╪════════════╪══════════╡
│ e                 ┆ Float64 ┆ 1185354    ┆ 1.0        ┆ 1        │
│ 192.168.100.3     ┆ Float64 ┆ 1185354    ┆ 1.0        ┆ 1        │
│ e s               ┆ Float64 ┆ 1185354    ┆ 1.0        ┆ 1        │
│ tcp               ┆ Float64 ┆ 1185354    ┆ 1.0        ┆ 1        │
│ 192.168.100.150   ┆ Float64 ┆ 1185354    ┆ 1.0        ┆ 1        │
│ …                 ┆ …       ┆ …          ┆ …          ┆ …        │
│ 0_duplicated_8    ┆ Float64 ┆ 1158918    ┆ 0.977698   ┆ 2        │
│ 19_duplicated_0   ┆ Float64 ┆ 1156086    ┆ 0.975309   ┆ 8        │
│ 1140_duplicated_0 ┆ Float64 ┆ 1156086    ┆ 0.975309   ┆ 8        │
│ 154_duplicated_0 

In [30]:
target_candidates = ["attack", "Attack", "label", "Label", "class", "Class", "attack_cat", "category"]
target_col = next((c for c in target_candidates if c in df_work.columns), None)

print("Detected target column:", target_col)

if target_col is None:
    raise RuntimeError("No target column found from known candidates.")

df_target = df_work.with_columns(pl.col(target_col).cast(pl.String).alias(target_col))
target_non_null = df_target.filter(pl.col(target_col).is_not_null())

target_dist = (
    target_non_null
    .group_by(target_col)
    .count()
    .rename({"count": "rows"})
    .sort("rows", descending=True)
)

print(target_dist.head(30))
print("Target non-null rows:", target_non_null.height, "out of", df_target.height)

Detected target column: attack
shape: (2, 2)
┌────────┬───────┐
│ attack ┆ rows  │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ 1      ┆ 87796 │
│ 0      ┆ 8     │
└────────┴───────┘
Target non-null rows: 87804 out of 1185354


In [31]:
import plotly.express as px

miss_pd = quality_df2.select(["column", "null_ratio"]).to_pandas().head(20)
fig1 = px.bar(miss_pd, x="column", y="null_ratio", title="Top 20 Missing Ratio (After Cleaning)")
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()

td_pd = target_dist.to_pandas()
fig2 = px.bar(td_pd.head(20), x=target_col, y="rows", title="Target Distribution (Non-null)")
fig2.show()

num_cols = []
for c, t in df_work.schema.items():
    if t in [pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8, pl.UInt64, pl.UInt32, pl.UInt16, pl.UInt8]:
        num_cols.append(c)

sentinel_rows = []
for c in num_cols:
    s = df_work.get_column(c)
    neg1 = int((s == -1).sum())
    zero = int((s == 0).sum())
    sentinel_rows.append({
        "column": c,
        "count_-1": neg1,
        "ratio_-1": neg1 / max(df_work.height, 1),
        "count_0": zero,
        "ratio_0": zero / max(df_work.height, 1),
    })

sentinel_df = pd.DataFrame(sentinel_rows).sort_values("ratio_-1", ascending=False)
print(sentinel_df.head(20))

               column  count_-1  ratio_-1  count_0       ratio_0
1               sport        10  0.000008        3  2.530890e-06
2               dport        10  0.000008        1  8.436298e-07
0             pkSeqID         0  0.000000        0  0.000000e+00
3                 seq         0  0.000000        0  0.000000e+00
4              stddev         0  0.000000    17491  1.475593e-02
5   N_IN_Conn_P_SrcIP         0  0.000000        0  0.000000e+00
6                 min         0  0.000000    50703  4.277456e-02
7        state_number         0  0.000000        0  0.000000e+00
8                mean         0  0.000000    15947  1.345336e-02
9   N_IN_Conn_P_DstIP         0  0.000000        0  0.000000e+00
10              drate         0  0.000000    81919  6.910931e-02
11              srate         0  0.000000     1275  1.075628e-03
12                max         0  0.000000    15947  1.345336e-02
13              stime         0  0.000000        0  0.000000e+00
14        flgs_number    

In [32]:
corr_ready_cols = []
for c in num_cols:
    nr = df_work.get_column(c).null_count() / max(df_work.height, 1)
    if nr < 0.30:
        corr_ready_cols.append(c)

# pick top variance columns
var_map = {}
if corr_ready_cols:
    v = df_work.select([pl.col(c).var().alias(c) for c in corr_ready_cols]).to_dicts()[0]
    for k, val in v.items():
        if val is not None:
            var_map[k] = float(val)

top_corr_cols = [k for k, _ in sorted(var_map.items(), key=lambda x: x[1], reverse=True)[:20]]

if len(top_corr_cols) >= 2:
    corr_sample = df_work.select(top_corr_cols).sample(n=min(200000, df_work.height), seed=42).to_pandas()
    corr_mat = corr_sample.corr(numeric_only=True)
    fig3 = px.imshow(corr_mat, title="Correlation Heatmap (Top Numeric)")
    fig3.show()
else:
    print("Not enough numeric columns for stable correlation heatmap.")

if "proto" in df_work.columns:
    ap = (
        df_target
        .filter(pl.col(target_col).is_not_null() & pl.col("proto").is_not_null())
        .group_by([target_col, "proto"])
        .count()
        .rename({"count": "rows"})
        .sort("rows", descending=True)
    )
    print("Attack vs Proto top rows:")
    print(ap.head(30))

if "state" in df_work.columns:
    as_ = (
        df_target
        .filter(pl.col(target_col).is_not_null() & pl.col("state").is_not_null())
        .group_by([target_col, "state"])
        .count()
        .rename({"count": "rows"})
        .sort("rows", descending=True)
    )
    print("Attack vs State top rows:")
    print(as_.head(30))

Attack vs Proto top rows:
shape: (6, 3)
┌────────┬───────┬───────┐
│ attack ┆ proto ┆ rows  │
│ ---    ┆ ---   ┆ ---   │
│ str    ┆ str   ┆ u32   │
╞════════╪═══════╪═══════╡
│ 1      ┆ udp   ┆ 45274 │
│ 1      ┆ tcp   ┆ 42443 │
│ 1      ┆ icmp  ┆ 69    │
│ 1      ┆ arp   ┆ 10    │
│ 0      ┆ udp   ┆ 6     │
│ 0      ┆ tcp   ┆ 2     │
└────────┴───────┴───────┘
Attack vs State top rows:
shape: (5, 3)
┌────────┬───────┬───────┐
│ attack ┆ state ┆ rows  │
│ ---    ┆ ---   ┆ ---   │
│ str    ┆ str   ┆ u32   │
╞════════╪═══════╪═══════╡
│ 1      ┆ INT   ┆ 29268 │
│ 1      ┆ REQ   ┆ 20723 │
│ 1      ┆ RST   ┆ 8491  │
│ 1      ┆ ACC   ┆ 46    │
│ 1      ┆ CON   ┆ 8     │
└────────┴───────┴───────┘


In [33]:
# time trend
if "stime" in df_work.columns:
    tdf = df_work.with_columns(
        pl.col("stime").cast(pl.Float64, strict=False).alias("stime_num")
    ).filter(pl.col("stime_num").is_not_null())

    # assume unix seconds if values are large
    tdf = tdf.with_columns(
        pl.from_epoch(pl.col("stime_num").cast(pl.Int64), time_unit="s").alias("event_time")
    )

    by_hour = (
        tdf
        .with_columns(pl.col("event_time").dt.truncate("1h").alias("hour"))
        .group_by("hour")
        .count()
        .rename({"count": "rows"})
        .sort("hour")
    )

    fig4 = px.line(by_hour.to_pandas(), x="hour", y="rows", title="Traffic Volume by Hour")
    fig4.show()

# top entities
for col in ["proto", "state", "saddr", "daddr", "sport", "dport"]:
    if col in df_work.columns:
        topv = (
            df_work
            .filter(pl.col(col).is_not_null())
            .group_by(col)
            .count()
            .rename({"count": "rows"})
            .sort("rows", descending=True)
            .head(20)
        )
        print(f"\nTop 20 {col}:")
        print(topv)


Top 20 proto:
shape: (4, 2)
┌───────┬───────┐
│ proto ┆ rows  │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ udp   ┆ 45280 │
│ tcp   ┆ 42445 │
│ icmp  ┆ 69    │
│ arp   ┆ 10    │
└───────┴───────┘

Top 20 state:
shape: (5, 2)
┌───────┬───────┐
│ state ┆ rows  │
│ ---   ┆ ---   │
│ str   ┆ u32   │
╞═══════╪═══════╡
│ INT   ┆ 29268 │
│ REQ   ┆ 20723 │
│ RST   ┆ 8491  │
│ ACC   ┆ 46    │
│ CON   ┆ 8     │
└───────┴───────┘

Top 20 saddr:
shape: (8, 2)
┌─────────────────┬───────┐
│ saddr           ┆ rows  │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞═════════════════╪═══════╡
│ 192.168.100.147 ┆ 35073 │
│ 192.168.100.148 ┆ 22368 │
│ 192.168.100.150 ┆ 15651 │
│ 192.168.100.149 ┆ 14593 │
│ 192.168.100.3   ┆ 70    │
│ 192.168.100.5   ┆ 42    │
│ 192.168.100.6   ┆ 4     │
│ 192.168.100.7   ┆ 3     │
└─────────────────┴───────┘

Top 20 daddr:
shape: (14, 2)
┌─────────────────┬───────┐
│ daddr           ┆ rows  │
│ ---             ┆ ---   │
│ str             ┆ u32   │
╞

In [35]:
out_dir = Path("/kaggle/working/eda_final_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

quality_df2.write_csv(out_dir / "quality_after_cleaning.csv")
target_dist.write_csv(out_dir / "target_distribution_non_null.csv")

if "sentinel_df" in globals():
    sentinel_df.to_csv(out_dir / "sentinel_report.csv", index=False)

# save light sample for report/debug
df_work.head(5000).write_csv(out_dir / "sample_head_5000.csv")

summary_text = []
summary_text.append(f"rows={df_work.height}, cols={df_work.width}")
summary_text.append(f"target_col={target_col}")
summary_text.append(f"target_non_null_rows={target_non_null.height}")
summary_text.append("outputs=quality_after_cleaning.csv,target_distribution_non_null.csv,sentinel_report.csv,sample_head_5000.csv")

with open(out_dir / "eda_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_text))

print("Saved:", out_dir)

Saved: /kaggle/working/eda_final_outputs


In [36]:
from pathlib import Path
import pandas as pd
import numpy as np

candidate_dirs = [
    Path("/kaggle/working/eda_final_outputs"),
    Path("/kaggle/working/eda_full_outputs"),
    Path("/kaggle/working/eda_fast_outputs"),
]

out_dir = None
for d in candidate_dirs:
    if d.exists():
        out_dir = d
        break

if out_dir is None:
    raise FileNotFoundError("No EDA output directory found in /kaggle/working")

print("Using output directory:", out_dir)

files = sorted(out_dir.glob("*"))
for f in files:
    print("-", f.name)

Using output directory: /kaggle/working/eda_final_outputs
- eda_summary.txt
- quality_after_cleaning.csv
- sample_head_5000.csv
- sentinel_report.csv
- target_distribution_non_null.csv


In [37]:
schema_path = out_dir / "quality_after_cleaning.csv"
if not schema_path.exists():
    schema_path = out_dir / "quality_summary.csv"

target_path = out_dir / "target_distribution_non_null.csv"
if not target_path.exists():
    target_path = out_dir / "target_distribution.csv"

sentinel_path = out_dir / "sentinel_report.csv"

schema_df = pd.read_csv(schema_path)
target_df = pd.read_csv(target_path) if target_path.exists() else None
sentinel_df = pd.read_csv(sentinel_path) if sentinel_path.exists() else None

total_cols = len(schema_df)
high_missing_50 = int((schema_df["null_ratio"] >= 0.50).sum()) if "null_ratio" in schema_df.columns else None
high_missing_98 = int((schema_df["null_ratio"] >= 0.98).sum()) if "null_ratio" in schema_df.columns else None

if target_df is not None and "rows" in target_df.columns:
    total_target_rows = int(target_df["rows"].sum())
    max_class = int(target_df["rows"].max())
    min_class = int(target_df["rows"].min()) if len(target_df) > 0 else 0
    imbalance_ratio = (max_class / min_class) if min_class > 0 else np.nan
else:
    total_target_rows = None
    imbalance_ratio = np.nan

if sentinel_df is not None and "ratio_-1" in sentinel_df.columns:
    top_sentinel = sentinel_df.sort_values("ratio_-1", ascending=False).head(10)
else:
    top_sentinel = None

print("Total columns:", total_cols)
print("Columns with missing >= 50%:", high_missing_50)
print("Columns with missing >= 98%:", high_missing_98)
print("Target rows used:", total_target_rows)
print("Class imbalance ratio (max/min):", round(imbalance_ratio, 2) if pd.notna(imbalance_ratio) else "N/A")

if target_df is not None:
    print("\nTop target classes:")
    print(target_df.head(10))

if top_sentinel is not None:
    print("\nTop sentinel (-1) columns:")
    print(top_sentinel[["column", "ratio_-1"]].head(10))

Total columns: 101
Columns with missing >= 50%: 97
Columns with missing >= 98%: 20
Target rows used: 87804
Class imbalance ratio (max/min): 10974.5

Top target classes:
   attack   rows
0       1  87796
1       0      8

Top sentinel (-1) columns:
              column  ratio_-1
0              sport  0.000008
1              dport  0.000008
2            pkSeqID  0.000000
3                seq  0.000000
4             stddev  0.000000
5  N_IN_Conn_P_SrcIP  0.000000
6                min  0.000000
7       state_number  0.000000
8               mean  0.000000
9  N_IN_Conn_P_DstIP  0.000000


In [38]:
report_lines = []
report_lines.append("EDA Report Summary")
report_lines.append("=" * 60)
report_lines.append(f"Total columns analyzed: {total_cols}")
report_lines.append(f"Columns with missing >= 50%: {high_missing_50}")
report_lines.append(f"Columns with missing >= 98%: {high_missing_98}")
report_lines.append(f"Target rows analyzed: {total_target_rows}")
report_lines.append(f"Class imbalance ratio (max/min): {round(imbalance_ratio, 2) if pd.notna(imbalance_ratio) else 'N/A'}")
report_lines.append("")

report_lines.append("Key Findings")
report_lines.append("- High-missing columns should be dropped or heavily imputed based on threshold.")
report_lines.append("- Target distribution indicates potential class imbalance risk.")
report_lines.append("- Sentinel values like -1 should be treated as special missing/invalid values for selected features.")
report_lines.append("- Correlated numeric features may require feature selection before modeling.")
report_lines.append("")

if target_df is not None:
    report_lines.append("Top Classes")
    for _, r in target_df.head(10).iterrows():
        report_lines.append(f"- {r.iloc[0]}: {int(r['rows'])}")
    report_lines.append("")

if top_sentinel is not None:
    report_lines.append("Top Sentinel(-1) Columns")
    for _, r in top_sentinel.head(10).iterrows():
        report_lines.append(f"- {r['column']}: {r['ratio_-1']:.4f}")
    report_lines.append("")

report_lines.append("Recommended Next Steps")
report_lines.append("1. Remove columns with missing ratio >= 0.98")
report_lines.append("2. Handle sentinel -1 values as missing where domain-appropriate")
report_lines.append("3. Apply class balancing strategy for classification tasks")
report_lines.append("4. Remove highly correlated redundant features")

report_path = out_dir / "eda_executive_summary.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Saved report:", report_path)
print("\nPreview:\n")
print("\n".join(report_lines[:25]))

Saved report: /kaggle/working/eda_final_outputs/eda_executive_summary.txt

Preview:

EDA Report Summary
Total columns analyzed: 101
Columns with missing >= 50%: 97
Columns with missing >= 98%: 20
Target rows analyzed: 87804
Class imbalance ratio (max/min): 10974.5

Key Findings
- High-missing columns should be dropped or heavily imputed based on threshold.
- Target distribution indicates potential class imbalance risk.
- Sentinel values like -1 should be treated as special missing/invalid values for selected features.
- Correlated numeric features may require feature selection before modeling.

Top Classes
- 1: 87796
- 0: 8

Top Sentinel(-1) Columns
- sport: 0.0000
- dport: 0.0000
- pkSeqID: 0.0000
- seq: 0.0000
- stddev: 0.0000
- N_IN_Conn_P_SrcIP: 0.0000


In [39]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

if "df_work" in globals():
    base_df = df_work
elif "df" in globals():
    base_df = df
else:
    raise RuntimeError("No dataframe found. Run previous EDA loading cells first.")

if "target_col" not in globals() or target_col is None:
    target_candidates = ["attack", "Attack", "label", "Label", "class", "Class", "attack_cat", "category"]
    target_col = next((c for c in target_candidates if c in base_df.columns), None)

print("Using target_col:", target_col)

# Plot sample to keep rendering fast
plot_n = min(250000, base_df.height)
plot_df = base_df.sample(n=plot_n, seed=42) if base_df.height > plot_n else base_df.clone()
print("Plot dataframe shape:", plot_df.shape)

Using target_col: attack
Plot dataframe shape: (250000, 101)


In [40]:
if target_col is not None:
    td = (
        plot_df
        .with_columns(pl.col(target_col).cast(pl.String))
        .filter(pl.col(target_col).is_not_null())
        .group_by(target_col)
        .count()
        .rename({"count": "rows"})
        .sort("rows", descending=True)
    )

    td_pd = td.to_pandas()
    td_pd["ratio"] = td_pd["rows"] / td_pd["rows"].sum()

    fig1 = px.bar(td_pd.head(20), x=target_col, y="rows", title="Target Class Distribution")
    fig1.show()

    fig2 = px.pie(td_pd.head(10), names=target_col, values="rows", title="Top 10 Target Classes")
    fig2.show()

    print(td_pd.head(20))
else:
    print("Target column not found.")

  attack   rows     ratio
0      1  18577  0.999892
1      0      2  0.000108


In [41]:
missing_plot = pl.DataFrame({
    "column": plot_df.columns,
    "null_ratio": [plot_df.get_column(c).null_count() / max(plot_df.height, 1) for c in plot_df.columns]
}).sort("null_ratio", descending=True)

mpd = missing_plot.to_pandas().head(30)

fig3 = px.bar(
    mpd,
    x="column",
    y="null_ratio",
    title="Top 30 Columns by Missing Ratio"
)
fig3.update_layout(xaxis_tickangle=-50)
fig3.show()

print(mpd.head(30))

               column  null_ratio
0                   e    1.000000
1       192.168.100.3    1.000000
2                 e s    1.000000
3                 tcp    1.000000
4     192.168.100.150    1.000000
5       192.168.100.6    1.000000
6                 REQ    1.000000
7                 DoS    1.000000
8                 TCP    1.000000
9     192.168.100.148    1.000000
10      192.168.100.5    1.000000
11                RST    1.000000
12                e g    1.000000
13    192.168.100.147    1.000000
14      192.168.100.7    1.000000
15                udp    1.000000
16    192.168.100.149    1.000000
17                INT    1.000000
18                UDP    1.000000
19               DDoS    1.000000
20     0_duplicated_8    0.977692
21    19_duplicated_0    0.975556
22  1140_duplicated_0    0.975556
23   154_duplicated_0    0.975144
24     0_duplicated_7    0.975144
25   308_duplicated_0    0.963504
26   770_duplicated_0    0.962876
27    11_duplicated_0    0.962736
28   660_dupli

In [42]:
cat_cols = [c for c, t in plot_df.schema.items() if t == pl.String and c not in ["_source_file"]]
print("Categorical columns:", cat_cols[:15])

for c in cat_cols[:4]:
    vc = (
        plot_df
        .filter(pl.col(c).is_not_null())
        .group_by(c)
        .count()
        .rename({"count": "rows"})
        .sort("rows", descending=True)
        .head(15)
    )
    vpd = vc.to_pandas()
    fig = px.bar(vpd, x=c, y="rows", title=f"Top 15 Values: {c}")
    fig.show()

Categorical columns: ['proto', 'saddr', 'daddr', 'attack', 'category', 'subcategory', 'flgs', 'state']


In [44]:
num_cols = [c for c, t in plot_df.schema.items() if t in [
    pl.Float64, pl.Float32, pl.Int64, pl.Int32, pl.Int16, pl.Int8, pl.UInt64, pl.UInt32, pl.UInt16, pl.UInt8
]]

# keep non-mostly-null numeric cols
num_cols = [c for c in num_cols if (plot_df.get_column(c).null_count() / max(plot_df.height, 1)) < 0.30]
print("Usable numeric columns:", len(num_cols))

hist_cols = num_cols[:8]
pdf = plot_df.select(hist_cols).to_pandas()

for c in hist_cols:
    fig = px.histogram(pdf, x=c, nbins=60, title=f"Distribution: {c}")
    fig.show()

Usable numeric columns: 2


In [45]:
if target_col is not None and len(num_cols) > 0:
    box_cols = num_cols[:6]
    bdf = (
        plot_df
        .select([target_col] + box_cols)
        .with_columns(pl.col(target_col).cast(pl.String))
        .filter(pl.col(target_col).is_not_null())
        .to_pandas()
    )

    top_classes = bdf[target_col].value_counts().head(5).index.tolist()
    bdf = bdf[bdf[target_col].isin(top_classes)]

    for c in box_cols:
        fig = px.box(bdf, x=target_col, y=c, title=f"{c} by {target_col}")
        fig.show()
else:
    print("Boxplot skipped: target or numeric columns not available.")

In [46]:
corr_cols = num_cols[:25]

if len(corr_cols) >= 2:
    cpdf = plot_df.select(corr_cols).to_pandas()
    corr = cpdf.corr(numeric_only=True)

    fig = px.imshow(corr, title="Correlation Heatmap (Top Numeric Columns)", aspect="auto")
    fig.show()

    pairs = []
    cols = corr.columns.tolist()
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            v = corr.iloc[i, j]
            if pd.notna(v) and abs(v) >= 0.85:
                pairs.append((cols[i], cols[j], float(v)))

    high_corr_df = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "corr"]).sort_values("corr", ascending=False)
    print("High-correlation pairs:", len(high_corr_df))
    print(high_corr_df.head(30))
else:
    print("Not enough columns for correlation.")

High-correlation pairs: 1
        feature_1       feature_2      corr
0  0_duplicated_0  0_duplicated_1  0.988889


In [47]:
if "stime" in plot_df.columns:
    tdf = (
        plot_df
        .with_columns(pl.col("stime").cast(pl.Float64, strict=False).alias("stime_num"))
        .filter(pl.col("stime_num").is_not_null())
        .with_columns(pl.from_epoch(pl.col("stime_num").cast(pl.Int64), time_unit="s").alias("event_time"))
        .with_columns(pl.col("event_time").dt.truncate("1h").alias("hour"))
    )

    by_hour = (
        tdf.group_by("hour")
        .count()
        .rename({"count": "rows"})
        .sort("hour")
    )

    fig = px.line(by_hour.to_pandas(), x="hour", y="rows", title="Traffic Volume by Hour")
    fig.show()

    if target_col is not None:
        by_hour_target = (
            tdf
            .with_columns(pl.col(target_col).cast(pl.String))
            .filter(pl.col(target_col).is_not_null())
            .group_by(["hour", target_col])
            .count()
            .rename({"count": "rows"})
            .sort("hour")
        )
        fig2 = px.line(by_hour_target.to_pandas(), x="hour", y="rows", color=target_col, title="Hourly Traffic by Target Class")
        fig2.show()
else:
    print("No stime column found.")

In [48]:
if target_col is not None:
    for c in ["proto", "state"]:
        if c in plot_df.columns:
            ctab = (
                plot_df
                .with_columns(pl.col(target_col).cast(pl.String), pl.col(c).cast(pl.String))
                .filter(pl.col(target_col).is_not_null() & pl.col(c).is_not_null())
                .group_by([target_col, c])
                .count()
                .rename({"count": "rows"})
                .sort("rows", descending=True)
            )

            cpd = ctab.to_pandas()
            top_targets = cpd[target_col].value_counts().head(8).index.tolist()
            top_c = cpd[c].value_counts().head(10).index.tolist()

            sub = cpd[cpd[target_col].isin(top_targets) & cpd[c].isin(top_c)]
            pivot = sub.pivot_table(index=target_col, columns=c, values="rows", aggfunc="sum", fill_value=0)

            fig = px.imshow(pivot, title=f"{target_col} vs {c} Heatmap", aspect="auto")
            fig.show()